# Chaos Control Comparison: α(x) vs OGY vs Pyragas

**Goal:** Quantitative comparison of α(x) static chaos control against the two standard methods:
- **OGY** (Ott-Grebogi-Yorke): small parameter perturbations near unstable periodic orbits
- **Pyragas** (delayed feedback): $x_{n+1} = f(x_n) + K(x_{n-\tau} - x_n)$
- **α(x)** (this work): static map modification $g(x) = \alpha(x) f(x) + (1-\alpha(x)) x$

**Metrics:** Basins of attraction, noise robustness, convergence speed, control effort.

**Predictions from Lyapunov results:**
- α(x) should have much larger basins (modifies map everywhere, not just near UPO)
- α(x) should be more noise-robust than Pyragas (no delay-feedback loop)
- α(x) will have higher "control effort" (entire map reshaped)
- The Lyapunov reflection symmetry Λ(α) = Λ(−α) means ±α give equivalent stabilization

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from petrification.maps import logistic
from petrification.iteration import iterate, iterate_transformed, find_fixed_points

plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 12})

## §2. Control Method Implementations

All three methods target the same goal: stabilize the fixed point $x^* = 1 - 1/a$ of the logistic map at parameter values where it's naturally unstable ($a > 3$).

In [ ]:
# ─── Fixed point and stability info ───
def fixed_point(a):
    """Non-trivial fixed point of logistic map."""
    return 1 - 1/a

def logistic_derivative(a, x):
    """f'(x) = a(1 - 2x)."""
    return a * (1 - 2*x)

# ─── OGY control ───
def ogy_iterate(a_nom, x0, n_iter, delta_max=0.01):
    """
    OGY control: perturb parameter a near the UPO.
    When |x - x*| < threshold, adjust a slightly to place x* on trajectory.
    
    For the logistic map x* = 1-1/a, the local dynamics near x* give:
      delta_a = (f'(x*) * (x_n - x*)) / (df/da at x*)
    where df/da = x*(1-x*) at the fixed point.
    """
    traj = np.zeros(n_iter + 1)
    control = np.zeros(n_iter)  # parameter perturbations applied
    traj[0] = x0
    x_star = fixed_point(a_nom)
    
    for i in range(n_iter):
        x = traj[i]
        dx = x - x_star
        
        # OGY: linearize near x* and compute needed parameter shift
        fp = logistic_derivative(a_nom, x_star)  # f'(x*) = 2 - a
        dfda = x_star * (1 - x_star)  # df/da at x*
        
        if abs(dx) < 0.1 and abs(dfda) > 1e-10:  # activation zone
            delta_a = -fp * dx / dfda
            delta_a = np.clip(delta_a, -delta_max, delta_max)
        else:
            delta_a = 0.0  # outside activation zone — no control
        
        control[i] = delta_a
        traj[i+1] = logistic(a_nom + delta_a, x)
    
    return traj, control

# ─── Pyragas (delayed feedback) ───
def pyragas_iterate(a, x0, n_iter, K=0.3, tau=1):
    """
    Pyragas delayed feedback: x_{n+1} = f(a, x_n) + K*(x_{n-tau} - x_n).
    Stabilizes period-tau orbits. For fixed point, tau=1.
    """
    traj = np.zeros(n_iter + 1)
    control = np.zeros(n_iter)
    traj[0] = x0
    # Fill initial delay buffer
    for i in range(min(tau, n_iter)):
        traj[i+1] = logistic(a, traj[i])
    
    for i in range(tau, n_iter):
        x = traj[i]
        x_delayed = traj[i - tau]
        feedback = K * (x_delayed - x)
        control[i] = feedback
        traj[i+1] = logistic(a, x) + feedback
        # Clamp to [0, 1] to prevent escape
        traj[i+1] = np.clip(traj[i+1], 0.001, 0.999)
    
    return traj, control

# ─── Alpha(x) control ───
def alpha_gaussian(x, x_star, alpha_ss, sigma=0.1):
    """Gaussian bump alpha profile: alpha=1 away from x*, alpha_ss at x*."""
    return 1 - (1 - alpha_ss) * np.exp(-(x - x_star)**2 / (2 * sigma**2))

def alpha_iterate(a, x0, n_iter, alpha_func):
    """Iterate with position-dependent alpha."""
    traj = iterate_transformed(logistic, alpha_func, a, x0, n_iter)
    # Control effort: how much alpha deviates from 1
    control = np.array([abs(alpha_func(traj[i]) - 1) for i in range(n_iter)])
    return traj, control

print('Control methods defined.')

## §3. Head-to-Head Comparison at a = 3.8

The logistic map at $a = 3.8$ is chaotic (Λ ≈ +0.44). All three methods attempt to stabilize the fixed point $x^* = 1 - 1/3.8 \approx 0.737$.

In [ ]:
a = 3.8
x_star = fixed_point(a)
n_iter = 200
x0 = 0.2  # far from fixed point

# Superstable alpha: g'(x*) = 0 when alpha(x*) = 1/(1 - f'(x*))
# f'(x*) = a(1-2x*) = a(1-2(1-1/a)) = a(2/a - 1) = 2-a
fp_at_star = logistic_derivative(a, x_star)
alpha_ss = 1 / (1 - fp_at_star + 1e-10)  # 1/(1-f') but f'=2-a, so 1/(a-1)
print(f'a = {a}, x* = {x_star:.6f}, f\'(x*) = {fp_at_star:.3f}')
print(f'Superstable alpha = {alpha_ss:.6f}')

alpha_func = lambda x: alpha_gaussian(x, x_star, alpha_ss, sigma=0.1)

# Run all three
traj_ogy, ctrl_ogy = ogy_iterate(a, x0, n_iter, delta_max=0.05)
traj_pyr, ctrl_pyr = pyragas_iterate(a, x0, n_iter, K=0.5, tau=1)
traj_alp, ctrl_alp = alpha_iterate(a, x0, n_iter, alpha_func)

# Uncontrolled reference
traj_none = iterate(logistic, a, x0, n_iter)

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Trajectories
axes[0].plot(traj_none, 'gray', alpha=0.3, linewidth=0.5, label='Uncontrolled')
axes[0].plot(traj_ogy, 'b-', linewidth=1, label='OGY', alpha=0.8)
axes[0].plot(traj_pyr, 'r-', linewidth=1, label='Pyragas', alpha=0.8)
axes[0].plot(traj_alp, 'g-', linewidth=1, label='α(x)', alpha=0.8)
axes[0].axhline(x_star, color='k', linestyle='--', alpha=0.3)
axes[0].set(ylabel='x_n', title=f'Chaos Control Comparison (a={a}, x*={x_star:.4f})')
axes[0].legend()

# Error from fixed point
for traj, label, color in [(traj_ogy, 'OGY', 'b'), (traj_pyr, 'Pyragas', 'r'), (traj_alp, 'α(x)', 'g')]:
    err = np.abs(traj - x_star)
    axes[1].semilogy(err, color=color, linewidth=1, label=label, alpha=0.8)
axes[1].set(ylabel='|x_n - x*|', title='Convergence to Fixed Point')
axes[1].legend()

# Control effort
axes[2].plot(np.abs(ctrl_ogy), 'b-', linewidth=0.5, label='OGY (|δa|)', alpha=0.8)
axes[2].plot(np.abs(ctrl_pyr), 'r-', linewidth=0.5, label='Pyragas (|feedback|)', alpha=0.8)
axes[2].plot(ctrl_alp, 'g-', linewidth=0.5, label='α(x) (|α-1|)', alpha=0.8)
axes[2].set(xlabel='Iteration', ylabel='Control magnitude', title='Control Effort')
axes[2].legend()

plt.tight_layout()
plt.show()

# Summary statistics
for name, traj in [('OGY', traj_ogy), ('Pyragas', traj_pyr), ('α(x)', traj_alp)]:
    err = np.abs(traj - x_star)
    converged = np.where(err < 1e-6)[0]
    first_conv = converged[0] if len(converged) > 0 else n_iter
    final_err = err[-1]
    print(f'{name:8s}: first convergence at iter {first_conv:4d}, final error = {final_err:.2e}')

## §4. Basins of Attraction

**Key test:** For a grid of initial conditions $x_0 \in [0, 1]$, which method successfully stabilizes to $x^*$?

This is where α(x) should dominate: since it modifies the map structure globally, convergence should be independent of $x_0$. OGY requires the trajectory to pass near $x^*$ naturally before control activates.

In [ ]:
# Basin of attraction analysis
a_values = [3.5, 3.7, 3.8, 3.9, 4.0]
x0_grid = np.linspace(0.01, 0.99, 200)
n_iter_basin = 500
tol_converged = 1e-4

fig, axes = plt.subplots(len(a_values), 1, figsize=(14, 3*len(a_values)), sharex=True)

basin_summary = {}

for ax, a in zip(axes, a_values):
    x_star = fixed_point(a)
    fp_val = logistic_derivative(a, x_star)
    alpha_ss = 1 / (1 - fp_val + 1e-10)
    alpha_func_local = lambda x, xs=x_star, ass=alpha_ss: alpha_gaussian(x, xs, ass, sigma=0.1)
    
    conv_ogy = np.zeros(len(x0_grid))
    conv_pyr = np.zeros(len(x0_grid))
    conv_alp = np.zeros(len(x0_grid))
    
    for j, x0 in enumerate(x0_grid):
        # OGY
        traj, _ = ogy_iterate(a, x0, n_iter_basin, delta_max=0.05)
        conv_ogy[j] = 1 if np.abs(traj[-1] - x_star) < tol_converged else 0
        
        # Pyragas
        traj, _ = pyragas_iterate(a, x0, n_iter_basin, K=0.5, tau=1)
        conv_pyr[j] = 1 if np.abs(traj[-1] - x_star) < tol_converged else 0
        
        # Alpha(x)
        traj, _ = alpha_iterate(a, x0, n_iter_basin, alpha_func_local)
        conv_alp[j] = 1 if np.abs(traj[-1] - x_star) < tol_converged else 0
    
    # Plot basins as colored bars
    y_ogy = np.where(conv_ogy, 0.9, np.nan)
    y_pyr = np.where(conv_pyr, 0.5, np.nan)
    y_alp = np.where(conv_alp, 0.1, np.nan)
    
    ax.scatter(x0_grid, y_ogy, c='blue', s=2, label=f'OGY ({conv_ogy.sum():.0f}/{len(x0_grid)})')
    ax.scatter(x0_grid, y_pyr, c='red', s=2, label=f'Pyragas ({conv_pyr.sum():.0f}/{len(x0_grid)})')
    ax.scatter(x0_grid, y_alp, c='green', s=2, label=f'α(x) ({conv_alp.sum():.0f}/{len(x0_grid)})')
    ax.set(ylabel=f'a={a}', ylim=(-0.1, 1.1), yticks=[])
    ax.legend(loc='upper right', fontsize=8)
    ax.axvline(x_star, color='k', linestyle='--', alpha=0.3)
    
    basin_summary[a] = {
        'OGY': conv_ogy.sum() / len(x0_grid),
        'Pyragas': conv_pyr.sum() / len(x0_grid),
        'alpha': conv_alp.sum() / len(x0_grid)
    }

axes[-1].set_xlabel('Initial condition x₀')
axes[0].set_title('Basins of Attraction: Which x₀ values converge to x*?')
plt.tight_layout()
plt.show()

# Summary table
print(f'{"a":>5s}  {"OGY":>8s}  {"Pyragas":>8s}  {"α(x)":>8s}')
for a in a_values:
    s = basin_summary[a]
    print(f'{a:5.1f}  {s["OGY"]:8.1%}  {s["Pyragas"]:8.1%}  {s["alpha"]:8.1%}')

## §5. Noise Robustness

Add observation noise $\xi_n \sim N(0, \sigma^2)$ to each iteration. OGY and Pyragas both use the noisy state for control decisions; α(x) applies the same deterministic modification regardless.

**Prediction:** α(x) should degrade gracefully since it doesn't rely on state estimation, while OGY (which needs to detect proximity to $x^*$) should fail first.

In [ ]:
def ogy_iterate_noisy(a_nom, x0, n_iter, delta_max=0.05, noise_std=0.0):
    """OGY with observation noise."""
    rng = np.random.default_rng(42)
    traj = np.zeros(n_iter + 1)
    traj[0] = x0
    x_star = fixed_point(a_nom)
    
    for i in range(n_iter):
        x = traj[i]
        x_observed = x + rng.normal(0, noise_std)  # noisy observation
        dx = x_observed - x_star
        fp = logistic_derivative(a_nom, x_star)
        dfda = x_star * (1 - x_star)
        
        if abs(dx) < 0.1 and abs(dfda) > 1e-10:
            delta_a = -fp * dx / dfda
            delta_a = np.clip(delta_a, -delta_max, delta_max)
        else:
            delta_a = 0.0
        
        x_next = logistic(a_nom + delta_a, x) + rng.normal(0, noise_std)
        traj[i+1] = np.clip(x_next, 0.001, 0.999)
    
    return traj

def pyragas_iterate_noisy(a, x0, n_iter, K=0.5, tau=1, noise_std=0.0):
    """Pyragas with observation noise."""
    rng = np.random.default_rng(42)
    traj = np.zeros(n_iter + 1)
    traj[0] = x0
    for i in range(min(tau, n_iter)):
        traj[i+1] = logistic(a, traj[i]) + rng.normal(0, noise_std)
        traj[i+1] = np.clip(traj[i+1], 0.001, 0.999)
    
    for i in range(tau, n_iter):
        x = traj[i]
        x_delayed = traj[i - tau]
        feedback = K * (x_delayed - x)
        x_next = logistic(a, x) + feedback + rng.normal(0, noise_std)
        traj[i+1] = np.clip(x_next, 0.001, 0.999)
    
    return traj

def alpha_iterate_noisy(a, x0, n_iter, alpha_func, noise_std=0.0):
    """Alpha(x) iteration with dynamical noise."""
    rng = np.random.default_rng(42)
    traj = np.zeros(n_iter + 1)
    traj[0] = x0
    for i in range(n_iter):
        x = traj[i]
        a_val = alpha_func(x)
        x_next = a_val * logistic(a, x) + (1 - a_val) * x + rng.normal(0, noise_std)
        traj[i+1] = np.clip(x_next, 0.001, 0.999)
    
    return traj

# Noise sweep
a = 3.8
x_star = fixed_point(a)
fp_val = logistic_derivative(a, x_star)
alpha_ss = 1 / (1 - fp_val + 1e-10)
alpha_func = lambda x: alpha_gaussian(x, x_star, alpha_ss, sigma=0.1)

noise_levels = np.logspace(-5, -1, 20)
n_trials = 50
n_iter_noise = 500

mean_err = {'OGY': [], 'Pyragas': [], 'alpha': []}

for sigma_n in noise_levels:
    errs = {'OGY': [], 'Pyragas': [], 'alpha': []}
    for trial in range(n_trials):
        x0 = np.random.default_rng(trial).uniform(0.1, 0.9)
        
        t_ogy = ogy_iterate_noisy(a, x0, n_iter_noise, noise_std=sigma_n)
        t_pyr = pyragas_iterate_noisy(a, x0, n_iter_noise, noise_std=sigma_n)
        t_alp = alpha_iterate_noisy(a, x0, n_iter_noise, alpha_func, noise_std=sigma_n)
        
        # Mean error over last 100 iterations
        errs['OGY'].append(np.mean(np.abs(t_ogy[-100:] - x_star)))
        errs['Pyragas'].append(np.mean(np.abs(t_pyr[-100:] - x_star)))
        errs['alpha'].append(np.mean(np.abs(t_alp[-100:] - x_star)))
    
    for method in mean_err:
        mean_err[method].append(np.median(errs[method]))

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(noise_levels, mean_err['OGY'], 'bo-', markersize=5, linewidth=1.5, label='OGY')
ax.loglog(noise_levels, mean_err['Pyragas'], 'rs-', markersize=5, linewidth=1.5, label='Pyragas')
ax.loglog(noise_levels, mean_err['alpha'], 'g^-', markersize=5, linewidth=1.5, label='α(x)')
ax.plot(noise_levels, noise_levels, 'k--', alpha=0.3, label='y = σ (noise floor)')
ax.set(xlabel='Noise std σ', ylabel='Median tracking error (last 100 iters)',
       title='Noise Robustness Comparison (a=3.8, 50 trials/level)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## §6. Convergence Speed Across Parameter Range

Sweep $a \in [3.0, 4.0]$ and measure how many iterations each method needs to reach $|x_n - x^*| < 10^{-8}$.

In [ ]:
a_sweep = np.linspace(3.01, 4.0, 80)
n_iter_speed = 2000
tol_speed = 1e-8
x0_speed = 0.2

speed_ogy = []
speed_pyr = []
speed_alp = []

for a in a_sweep:
    x_star = fixed_point(a)
    fp_val = logistic_derivative(a, x_star)
    alpha_ss = 1 / (1 - fp_val + 1e-10)
    alpha_func_local = lambda x, xs=x_star, ass=alpha_ss: alpha_gaussian(x, xs, ass, sigma=0.1)
    
    # OGY
    traj, _ = ogy_iterate(a, x0_speed, n_iter_speed, delta_max=0.05)
    err = np.abs(traj - x_star)
    conv_idx = np.where(err < tol_speed)[0]
    speed_ogy.append(conv_idx[0] if len(conv_idx) > 0 else n_iter_speed)
    
    # Pyragas
    traj, _ = pyragas_iterate(a, x0_speed, n_iter_speed, K=0.5, tau=1)
    err = np.abs(traj - x_star)
    conv_idx = np.where(err < tol_speed)[0]
    speed_pyr.append(conv_idx[0] if len(conv_idx) > 0 else n_iter_speed)
    
    # Alpha(x)
    traj, _ = alpha_iterate(a, x0_speed, n_iter_speed, alpha_func_local)
    err = np.abs(traj - x_star)
    conv_idx = np.where(err < tol_speed)[0]
    speed_alp.append(conv_idx[0] if len(conv_idx) > 0 else n_iter_speed)

fig, ax = plt.subplots(figsize=(12, 6))
ax.semilogy(a_sweep, speed_ogy, 'b.-', markersize=3, linewidth=1, label='OGY')
ax.semilogy(a_sweep, speed_pyr, 'r.-', markersize=3, linewidth=1, label='Pyragas')
ax.semilogy(a_sweep, speed_alp, 'g.-', markersize=3, linewidth=1, label='α(x)')
ax.axhline(n_iter_speed, color='k', linestyle='--', alpha=0.3, label='Did not converge')
ax.set(xlabel='Parameter a', ylabel='Iterations to |x-x*| < 10⁻⁸',
       title='Convergence Speed vs Bifurcation Parameter')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## §7. The Z₂ Symmetry Test

Our Lyapunov results showed $\Lambda(\alpha) = \Lambda(-\alpha)$. Test whether positive and negative α achieve identical stabilization.

In [ ]:
a = 3.9
x_star = fixed_point(a)
fp_val = logistic_derivative(a, x_star)
alpha_ss_pos = 1 / (1 - fp_val + 1e-10)
alpha_ss_neg = -alpha_ss_pos

alpha_pos = lambda x: alpha_gaussian(x, x_star, alpha_ss_pos, sigma=0.1)
alpha_neg = lambda x: alpha_gaussian(x, x_star, alpha_ss_neg, sigma=0.1)
alpha_const_pos = lambda x: 0.5
alpha_const_neg = lambda x: -0.5

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

configs = [
    (alpha_pos, f'Gaussian α(x*) = {alpha_ss_pos:.3f}', 'green'),
    (alpha_neg, f'Gaussian α(x*) = {alpha_ss_neg:.3f}', 'purple'),
    (alpha_const_pos, 'Constant α = 0.5', 'blue'),
    (alpha_const_neg, 'Constant α = -0.5', 'red'),
]

for ax, (af, label, color) in zip(axes.flatten(), configs):
    traj = iterate_transformed(logistic, af, a, 0.2, 150)
    err = np.abs(traj - x_star)
    ax.semilogy(err, color=color, linewidth=1.5)
    ax.set(title=label, ylabel='|x_n - x*|', xlabel='Iteration')
    ax.grid(True, alpha=0.3)
    # Report convergence
    final_err = err[-1]
    ax.text(0.95, 0.95, f'Final err: {final_err:.2e}', transform=ax.transAxes,
            ha='right', va='top', fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle(f'Z₂ Symmetry Test: α vs -α at a={a}', fontsize=14)
plt.tight_layout()
plt.show()

# Direct comparison of convergence rates
for af, label in [(alpha_const_pos, 'α=+0.5'), (alpha_const_neg, 'α=-0.5')]:
    traj = iterate_transformed(logistic, af, a, 0.2, 500)
    err = np.abs(traj - x_star)
    # Estimate Lyapunov from consecutive errors
    log_err = np.log(err[100:] + 1e-16)
    lyap_est = np.mean(np.diff(log_err))
    print(f'{label}: Lyapunov est ≈ {lyap_est:.4f}, final err = {err[-1]:.2e}')

## §8. Control Effort Comparison

Define a unified "control cost" metric:
- **OGY:** $C = \sum_n |\delta a_n|^2$ (total parameter perturbation energy)
- **Pyragas:** $C = \sum_n |K(x_{n-\tau} - x_n)|^2$ (total feedback energy)
- **α(x):** $C = \sum_n |\alpha(x_n) - 1|^2 \cdot |f(x_n) - x_n|^2$ (actual displacement from natural dynamics)

The last definition measures how much α(x) actually changes each iteration step, not just how far α is from 1.

In [ ]:
# Control cost across parameter range
a_cost_sweep = np.linspace(3.2, 4.0, 50)
n_iter_cost = 500
x0_cost = 0.2

cost_ogy = []
cost_pyr = []
cost_alp = []

for a in a_cost_sweep:
    x_star = fixed_point(a)
    fp_val = logistic_derivative(a, x_star)
    alpha_ss = 1 / (1 - fp_val + 1e-10)
    af = lambda x, xs=x_star, ass=alpha_ss: alpha_gaussian(x, xs, ass, sigma=0.1)
    
    traj_o, ctrl_o = ogy_iterate(a, x0_cost, n_iter_cost, delta_max=0.05)
    traj_p, ctrl_p = pyragas_iterate(a, x0_cost, n_iter_cost, K=0.5, tau=1)
    traj_a, ctrl_a = alpha_iterate(a, x0_cost, n_iter_cost, af)
    
    cost_ogy.append(np.sum(ctrl_o**2))
    cost_pyr.append(np.sum(ctrl_p**2))
    # For alpha: actual displacement from natural dynamics
    displacements = []
    for i in range(n_iter_cost):
        x = traj_a[i]
        natural = logistic(a, x)
        controlled = af(x) * logistic(a, x) + (1 - af(x)) * x
        displacements.append((controlled - natural)**2)
    cost_alp.append(np.sum(displacements))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.semilogy(a_cost_sweep, cost_ogy, 'b.-', markersize=4, label='OGY')
ax1.semilogy(a_cost_sweep, cost_pyr, 'r.-', markersize=4, label='Pyragas')
ax1.semilogy(a_cost_sweep, cost_alp, 'g.-', markersize=4, label='α(x)')
ax1.set(xlabel='Parameter a', ylabel='Total control cost', title='Control Effort (500 iters)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Efficiency: cost per unit convergence
eff_ogy = np.array(cost_ogy) / np.maximum(np.array([1]*len(cost_ogy)), 1)
ax2.plot(a_cost_sweep, np.array(cost_ogy) / (np.array(speed_ogy[:len(a_cost_sweep)]) + 1), 'b.-',
         markersize=4, label='OGY')
ax2.plot(a_cost_sweep, np.array(cost_pyr) / (np.array(speed_pyr[:len(a_cost_sweep)]) + 1), 'r.-',
         markersize=4, label='Pyragas')
ax2.plot(a_cost_sweep, np.array(cost_alp) / (np.array(speed_alp[:len(a_cost_sweep)]) + 1), 'g.-',
         markersize=4, label='α(x)')
ax2.set(xlabel='Parameter a', ylabel='Cost / convergence time', title='Cost-Efficiency')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## §9. Higher-Period Orbit Stabilization

Can α(x) stabilize period-2 orbits, not just fixed points? The Pyragas method naturally extends (set $\tau = 2$). OGY targets the period-2 UPO. For α(x), we need to target the period-2 orbit of $f$, which are fixed points of $f^2$.

In [ ]:
# Period-2 stabilization comparison
a = 3.9
n_iter_p2 = 300
x0 = 0.2

# Find period-2 orbit points
def f2(a, x):
    return logistic(a, logistic(a, x))

fps_f2 = find_fixed_points(f2, a, (0.01, 0.99), n_points=5000)
x_star_fp = fixed_point(a)
# Period-2 points are fixed points of f^2 that aren't fixed points of f
p2_points = [x for x in fps_f2 if abs(x - x_star_fp) > 0.01]
print(f'Period-2 orbit at a={a}: {[f"{x:.6f}" for x in p2_points]}')

if len(p2_points) >= 2:
    p2_a, p2_b = p2_points[0], p2_points[1]
    print(f'Verification: f({p2_a:.6f}) = {logistic(a, p2_a):.6f} ≈ {p2_b:.6f}')
    print(f'Verification: f({p2_b:.6f}) = {logistic(a, p2_b):.6f} ≈ {p2_a:.6f}')
    
    # Pyragas with tau=2
    traj_pyr2, _ = pyragas_iterate(a, x0, n_iter_p2, K=0.5, tau=2)
    
    # Alpha approach: stabilize f^2 at p2_a
    # g^2'(p2_a) = 0 when alpha chosen so that each step contracts
    # Simpler: use alpha on f^2 directly as the iteration
    f2_prime_at_p2 = logistic_derivative(a, p2_a) * logistic_derivative(a, p2_b)  # chain rule
    alpha_p2 = 1 / (1 - f2_prime_at_p2 + 1e-10)
    print(f'f^2\'(p2_a) = {f2_prime_at_p2:.4f}, alpha for superstable = {alpha_p2:.4f}')
    
    # Iterate f^2 with alpha-transform
    # But we want to iterate f (not f^2) with alpha — stabilizing the period-2 orbit
    # Use alpha that makes each pair of steps contract
    # For period-2: apply alpha near BOTH orbit points
    alpha_p2_func = lambda x: 1 - (1 - alpha_p2) * (
        np.exp(-(x - p2_a)**2 / (2*0.05**2)) + np.exp(-(x - p2_b)**2 / (2*0.05**2))
    )
    
    traj_alp2 = iterate_transformed(logistic, alpha_p2_func, a, x0, n_iter_p2)
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 7))
    
    axes[0].plot(traj_pyr2, 'r-', linewidth=1, label='Pyragas (τ=2)')
    axes[0].plot(traj_alp2, 'g-', linewidth=1, label='α(x) (dual Gaussian)')
    axes[0].axhline(p2_a, color='k', linestyle='--', alpha=0.3)
    axes[0].axhline(p2_b, color='k', linestyle=':', alpha=0.3)
    axes[0].set(ylabel='x_n', title=f'Period-2 Stabilization at a={a}')
    axes[0].legend()
    
    # Distance from period-2 orbit
    dist_pyr2 = np.minimum(np.abs(traj_pyr2 - p2_a), np.abs(traj_pyr2 - p2_b))
    dist_alp2 = np.minimum(np.abs(traj_alp2 - p2_a), np.abs(traj_alp2 - p2_b))
    
    axes[1].semilogy(dist_pyr2, 'r-', linewidth=1, label='Pyragas (τ=2)')
    axes[1].semilogy(dist_alp2, 'g-', linewidth=1, label='α(x)')
    axes[1].set(xlabel='Iteration', ylabel='Distance from period-2 orbit',
               title='Convergence to Period-2 Orbit')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print('Could not find period-2 orbit points.')

## §10. Assessment

| Metric | OGY | Pyragas | α(x) |
|--------|-----|---------|------|
| Basin of attraction | Fill after running | Fill after running | Fill after running |
| Noise robustness | Fill after running | Fill after running | Fill after running |
| Convergence speed | Fill after running | Fill after running | Fill after running |
| Control effort | Fill after running | Fill after running | Fill after running |
| Period-2 stabilization | N/A | Fill after running | Fill after running |
| Implementation complexity | Needs UPO detection | Needs delay buffer | Static map modification |
| Real-time requirement | Yes (detect crossings) | Yes (feedback loop) | No (predetermined) |

### Hints for higher dimensions

- **α → matrix α:** In 2D maps, $\alpha(\mathbf{x})$ becomes a $2 \times 2$ mixing matrix. The basin analysis would extend naturally.
- **Gauge structure:** If the optimal $\alpha(x)$ varies significantly across phase space, the gauge-theoretic formulation (§2 of cross_theory_connections) becomes relevant.
- **Koopman:** The Z₂ symmetry $\Lambda(\alpha) = \Lambda(-\alpha)$ should generalize to matrix maps with an involution symmetry.